In [1]:
print("Let's Start")

Let's Start


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error , r2_score

from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

In [4]:
X = df.drop(columns=['price'])
y = df['price']

In [16]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [23]:
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor , ExtraTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor , RandomForestRegressor , AdaBoostRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from xgboost import XGBRegressor

X = df.drop(columns=['price'])
y = df['price']

y_transformed = np.log1p(y)


# -----------------------------
# Categorical columns
# -----------------------------

ordinal_columns = [
    'property_type',
    'balcony',
    'furnishing_type',
    'luxury_category',
    'floor_category'
]


# -----------------------------
# Preprocessor
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[
        
        (
            'num',
            StandardScaler(),
            [
                'bedRoom',
                'bathroom',
                'built_up_area',
                'servant room',
                'store room'
            ]
        ),

        (
            'ordinal',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            ordinal_columns
        ),

        (
            'age_onehot',
            OneHotEncoder(
                drop='first',
                sparse_output=False,
                handle_unknown='ignore'
            ),
            ['agePossession']
        ),

        (
            'sector_target',
            TargetEncoder(),
            ['sector']
        )
    ],

    remainder='passthrough'
)


# -----------------------------
# XGBoost parameters
# -----------------------------

best_params = {
    'n_estimators': 237,
    'learning_rate': 0.019641970874481877,
    'max_depth': 10,
    'subsample': 0.5951790412625925
}


# -----------------------------
# Pipeline
# -----------------------------

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(**best_params))
])


# -----------------------------
# Train/test split
# -----------------------------

from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# -----------------------------
# Train
# -----------------------------

pipeline.fit(xtrain, ytrain)


# -----------------------------
# Predict
# -----------------------------

ypred = np.expm1(
    pipeline.predict(xtest)
)

In [21]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [24]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreeRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'xgboost':XGBRegressor()
}
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])


In [26]:
model_df.sort_values(by='mae')

,name,r2,mae
5,random forest,0.892340,0.461980
9,xgboost,0.898563,0.483924
7,gradient boosting,0.883468,0.526492
1,svr,0.863194,0.583774
4,decision tree,0.803927,0.590038
8,adaboost,0.817560,0.676179
6,extra trees,0.769494,0.704814
0,linear_reg,0.828997,0.714540
2,ridge,0.829011,0.715103
3,LASSO,-0.003598,1.576392


In [30]:
import optuna
import numpy as np

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# ============================================================
# DATA
# ============================================================

X = df.drop(columns=["price"])
y = np.log1p(df["price"])


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# OPTUNA OBJECTIVE
# ============================================================

def objective(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 15
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10, log=True
        ),

        "objective": "reg:squarederror",

        "random_state": 42,

        "n_jobs": -1
    }


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    model = XGBRegressor(**params)


    # --------------------------------------------------------
    # Pipeline
    # --------------------------------------------------------

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    y_pred = pipeline.predict(X_test)


    # --------------------------------------------------------
    # RMSE
    # --------------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    return rmse


# ============================================================
# OPTUNA STUDY
# ============================================================

study = optuna.create_study(
    direction="minimize"
)


study.optimize(
    objective,
    n_trials=50
)


# ============================================================
# RESULTS
# ============================================================

print("\nBest RMSE:")
print(study.best_value)

print("\nBest Parameters:")

for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")


Best RMSE:
0.1658058881138523

Best Parameters:
n_estimators: 974
learning_rate: 0.10526914056016663
max_depth: 12
min_child_weight: 1
gamma: 0.018795185277108897
subsample: 0.8079122956973779
colsample_bytree: 0.7947938787822193
reg_alpha: 4.17187378088139e-05
reg_lambda: 2.1089841197246224e-07


In [31]:
import optuna
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X = df.drop(columns=["price"])
y = np.log1p(df["price"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", 1.0]
        ),
        "bootstrap": trial.suggest_categorical(
            "bootstrap", [True, False]
        ),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse


# Create Optuna study
study = optuna.create_study(direction="minimize")

# Run tuning
study.optimize(objective, n_trials=50)


print("\nBest RMSE:")
print(study.best_value)

print("\nBest Parameters:")
for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")


# Train final model using best parameters
best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

final_pipeline.fit(X_train, y_train)


# Prediction
ypred = np.expm1(final_pipeline.predict(X_test))



Best RMSE:
0.16620849674896945

Best Parameters:
n_estimators: 401
max_depth: 27
min_samples_split: 2
min_samples_leaf: 1
max_features: sqrt
bootstrap: False


In [32]:
xgparams = {'n_estimators': 974,
'learning_rate': 0.10526914056016663,
'max_depth': 12,
'min_child_weight': 1,
'gamma': 0.018795185277108897,
'subsample': 0.8079122956973779,
'colsample_bytree': 0.7947938787822193,
'reg_alpha': 4.17187378088139e-05,
'reg_lambda': 2.1089841197246224e-07}

rfparams = study.best_params

In [33]:
model_dict = {
    'random forest':RandomForestRegressor(**rfparams),
    'xgboost':XGBRegressor(**xgparams)
}
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])


In [34]:
model_df.sort_values(by='mae')

,name,r2,mae
1,xgboost,0.895949,0.459995
0,random forest,0.892840,0.488877
